# Silver Layer — Data Cleaning & Transformation
### Notebook: 02_silver_transformation
**Pipeline:** Bank Customer Churn Analytics
**Author:** Harshkumar Patel
**Date:** 29/06/2026

---

## What this notebook does
Silver layer makes the raw bronze data usable by cleaning it,
renaming columns and splitting it into a star schema structure

## Silver Layer Rule
> Clean it. Structure it. Enrich it. But do not aggregate it.

## Source
- **Database:** churn_bronze
- **Table:** churn_bronze.raw_bank_customers
- **Rows:** 80,000

## Target Tables
- **fact_churn_events** — 1 row per customer, central churn data
- **dim_customer** — customer demographics
- **dim_geography** — province and location info
- **dim_financial** — balance, credit score, cards
- **dim_engagement** — activity, engagement score, loyalty

## What we do in this notebook
1. Read from Bronze table
2. Rename truncated column names
3. Drop PII columns (full_name, address)
4. Fix data types
5. Remove duplicates
6. Feature engineering — create new columns
7. Split into star schema tables

##1. Read from Bronze table

In [0]:
# loading data from the bronze delta table into silver for cleaning
df_bronze = spark.table("churn_bronze.raw_bank_customers")
print(f"Rows loaded from bronze table: {df_bronze.count():,}")

##Drop PII columns (full_name, address) And Irrelevent Columns
PII = Personally Identifiable Information. In our dataset two columns contain personal details that should never be in an analytics table:

full_name &
address

In [0]:
print(f"Number of Columns: {len(df_bronze.columns)}")
# drop columns that are not needed for the analysis
df_clean = df_bronze.drop('full_name', 'address','_ingestion_timestamp', '_source_file', '_pipeline_layer')
print(f"Number of Columns after dropping: {len(df_clean.columns)}")

##Rename truncated column names

In [0]:
df_clean = df_clean.withColumnRenamed('tenure_ye', 'tenure_year')
df_clean = df_clean.withColumnRenamed('credit_sco', 'credit_score')
df_clean = df_clean.withColumnRenamed('monthly_ir', 'monthly_income')
df_clean = df_clean.withColumnRenamed('nums_card', 'num_of_cards')
df_clean = df_clean.withColumnRenamed('nums_service', 'num_of_services')

print(df_clean.columns)

##Removing/ Dropping Duplicate Values

In [0]:
print(df_clean.count())
df_clean = df_clean.dropDuplicates(['id'])
print(df_clean.count())
# write the cleaned data to the silver delta table
# df_clean.write.mode("overwrite").saveAsTable("churn_silver.bank_customers"). # no need as there are no duplicate data.

## Fixing Data Types


In [0]:
from pyspark.sql.types import DoubleType
# Converting it to double because the column contains decimal values ot the balance can be in decimal values.
df_clean = df_clean.withColumn('balance', df_clean['balance'].cast(DoubleType()))

print(df_clean.printSchema())

##Feature engineering — create new columns

Feature : age_group + balance_tier

In [0]:
from pyspark.sql.functions import when

#Analyising the tenure column
print(df_clean.select('age').summary().display().max('age'))
#Dividing age into 3 catogories
df_clean = df_clean.withColumn('age_group',
    when(df_clean['age'] <= 30, 'Young')
    .when(df_clean['age'] <= 50, 'Middle Aged')
    .otherwise('Senior')
)

df_clean.select('age', 'age_group').display()

#Analyising the tenure column
print(df_clean.select('balance').summary().display().max('balance'))

#Dividing balance into 5 catogories

df_clean = df_clean.withColumn('balance_tier',
            when(df_clean['balance'] == 0, 'Zero Balance')
            .when(df_clean['balance'] <= 5000000, 'Low')
            .when(df_clean['balance'] <= 20000000, 'Medium')
            .when(df_clean['balance'] <= 50000000, 'High')
            .when(df_clean['balance'] > 50000000, 'Premium')
            .otherwise('Unknown')
        )
df_clean.select('balance', 'balance_tier').display()



Feature : tenure_group

In [0]:
#Analyising the tenure column
print(df_clean.select('tenure_year').summary().display().max('tenure_year'))
#Dividing tenure into 3 catogories
df_clean = df_clean.withColumn('tenure_group',
            when(df_clean['tenure_year'] <= 1, 'New')
            .when(df_clean['tenure_year'] <= 3, 'Regular')
            .otherwise('Loyal'))

df_clean.select('tenure_year', 'tenure_group').display()

df_clean.select('age', 'age_group').display()
df_clean.select('balance', 'balance_tier').display()
df_clean.select('tenure_year', 'tenure_group').display()